pip install -r requirements.txt

# Imports

In [ ]:
import requests
import json
import time
from datetime import date, timedelta
from pathlib import Path
from requests.exceptions import Timeout, ConnectionError, ReadTimeout

# Settings

In [8]:
BASE    = "https://www.atg.se/services/racinginfo/v1/api"
ROOT    = Path.cwd().parent.parent
RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Fetch JSON from api

In [ ]:
def fetch(url, max_retries=3, timeout=60):
    """Fetch JSON from URL with exponential backoff retry logic."""
    for attempt in range(max_retries):
        try:
            response = requests.get(url, headers={"Accept": "application/json"}, timeout=timeout)
            response.raise_for_status()
            return response.json()
        except (Timeout, ReadTimeout, ConnectionError) as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt  # exponential backoff: 1s, 2s, 4s
                print(f"    ⚠️  Timeout/connection error. Retrying in {wait_time}s... (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait_time)
            else:
                print(f"    ❌ Failed after {max_retries} attempts: {e}")
                raise

# Save to file

In [10]:
def save(filename, data):
    path = RAW_DIR / f"{filename}.json"
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2))

# Get a date, if date exist > skip fetch

In [11]:
def fetch_day(date_str):
    print(f"Fetching {date_str}...")

    calendar = fetch(f"{BASE}/calendar/day/{date_str}")
    save(f"calendar_{date_str}", calendar)

    for game_type, game_list in calendar.get("games", {}).items():
        if game_type not in {"V64", "V75", "V85", "V86"}:
            continue

        for game_stub in game_list:
            game_id   = game_stub["id"]
            game_path = RAW_DIR / f"game_{game_id}.json"

            if game_path.exists():
                print(f"  Skipping {game_id} (already exists)")
                continue

            game = fetch(f"{BASE}/games/{game_id}")
            save(f"game_{game_id}", game)

            for race in game.get("races", []):
                race_id   = race["id"]
                race_path = RAW_DIR / f"race_{race_id}.json"

                if race_path.exists():
                    print(f"    Skipping {race_id} (already exists)")
                    continue

                result = fetch(f"{BASE}/races/{race_id}")
                save(f"race_{race_id}", result)

# Run 

In [12]:
yesterday = date.today() - timedelta(days=1)

# Change range(365) to range(7) for a quick test, or range(365) for a full year
for i in range(7):
    day = (yesterday - timedelta(days=i)).isoformat()
    try:
        fetch_day(day)
    except Exception as e:
        print(f"⚠️  Skipping {day} due to error: {e}")
        continue

print("Done!")

Fetching 2026-05-19...
  Skipping V64_2026-05-19_25_4 (already exists)
Fetching 2026-05-18...
  Skipping V64_2026-05-18_13_4 (already exists)
Fetching 2026-05-17...
  Skipping V85_2026-05-17_55_5 (already exists)
  Skipping V64_2026-05-17_79_3 (already exists)
Fetching 2026-05-16...
  Skipping V85_2026-05-16_46_5 (already exists)
Fetching 2026-05-15...
Fetching 2026-05-14...
  Skipping V64_2026-05-14_7_5 (already exists)
Fetching 2026-05-13...
  Skipping V86_2026-05-13_40_1 (already exists)
Done!


## Konfiguration

Ändra `SPELTYPER` om du vill hämta fler eller färre spelformer. 